# Versão modular · mesma referência científica
As funções estão em src/hai_tcc; o notebook apresenta cada passo e seus resultados.
Configuração: configs/baseline.yaml. Saídas: runs/baseline_modular/.
Cada etapa começa em um kernel novo e lê os artefatos salvos em disco.


In [1]:
from pathlib import Path
import sys
for BASE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (BASE_DIR / "configs/baseline.yaml").is_file() and (BASE_DIR / "src/hai_tcc").is_dir():
        break
else:
    raise FileNotFoundError("Abra a pasta do projeto.")
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))
from hai_tcc.experiment import abrir_experimento
exp = abrir_experimento(BASE_DIR)
import numpy as np
import pandas as pd

print("Python utilizado:", sys.executable)
print("Versão do pandas:", pd.__version__)
print("Experimento:", exp.pasta)


Python utilizado: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Versão do pandas: 2.2.3
Experimento: /workspace/scratch/322cdfdc685a/tmp/tcc-hai_modular/runs/baseline_modular


In [2]:
import json
from hai_tcc.data import garantir_dados, resumir_arquivos, verificar_qualidade
from hai_tcc.features import colunas_industriais
origem = json.loads((BASE_DIR / "data/origem.json").read_text())
garantir_dados(exp.dados, origem)
arquivos = exp.config["arquivos"]


Conferido: test1.csv.gz
Conferido: test2.csv.gz
Conferido: test3.csv.gz
Conferido: test4.csv.gz
Conferido: test5.csv.gz
Conferido: train1.csv.gz
Conferido: train2.csv.gz
Conferido: train3.csv.gz


In [3]:
amostra = pd.read_csv(exp.dados / "train1.csv.gz", nrows=5)
print("Linhas da amostra:", len(amostra), "· Colunas:", amostra.shape[1])
with pd.option_context("display.max_rows", None):
    display(amostra.T)


Linhas da amostra: 5 · Colunas: 84


,0,1,2,3,4
time,2020-07-11 00:00:00,2020-07-11 00:00:01,2020-07-11 00:00:02,2020-07-11 00:00:03,2020-07-11 00:00:04
P1_B2004,0.10121,0.10121,0.10121,0.10121,0.10121
P1_B2016,1.29784,1.29692,1.29631,1.28685,1.28807
P1_B3004,397.63785,397.63785,397.63785,397.63785,397.63785
P1_B3005,1001.99799,1001.99799,1001.99799,1001.99799,1001.99799
P1_B4002,33.6555,33.6555,33.6555,33.6555,33.6555
P1_B4005,100,100,100,100,100
P1_B400B,2847.02539,2839.5852,2833.26807,2834.95264,2832.70654
P1_B4022,37.14706,37.14477,37.14325,37.11959,37.12265
P1_FCV01D,100,100,100,100,100


In [4]:
resumo_arquivos = resumir_arquivos(exp.dados, arquivos)
display(resumo_arquivos)
resumo_arquivos.to_csv(exp.resultados / "resumo_arquivos.csv", index=False)


,arquivo,registros,normais,ataques,início,fim
0,train1.csv.gz,216001,216001,0,2020-07-11 00:00:00,2020-07-13 12:00:00
1,train2.csv.gz,226801,226801,0,2020-07-31 22:00:00,2020-08-03 13:00:00
2,train3.csv.gz,478801,478801,0,2020-08-04 22:00:00,2020-08-10 11:00:00
3,test1.csv.gz,43201,42572,629,2020-07-07 15:00:00,2020-07-08 03:00:00
4,test2.csv.gz,118801,115352,3449,2020-07-09 15:00:00,2020-07-11 00:00:00
5,test3.csv.gz,108001,106466,1535,2020-07-13 00:00:00,2020-07-14 06:00:00
6,test4.csv.gz,39601,38444,1157,2020-07-28 12:00:00,2020-07-28 23:00:00
7,test5.csv.gz,92401,90224,2177,2020-07-30 10:50:00,2020-07-31 12:30:00


In [5]:
colunas_entrada = colunas_industriais(amostra.columns)
print("Quantidade de variáveis de entrada:", len(colunas_entrada))
print("Fora de X:", [c for c in amostra if c not in colunas_entrada])


Quantidade de variáveis de entrada: 79
Fora de X: ['time', 'attack', 'attack_P1', 'attack_P2', 'attack_P3']


7 Célula - Verificando se existe valores ausentes, horários repetidos ou rótulos inesperados nos arquivos.

In [6]:
resumo_qualidade = verificar_qualidade(exp.dados, arquivos, amostra.columns)
display(resumo_qualidade)
contagens = ["valores_ausentes", "horários_inválidos", "horários_repetidos", "rótulos_inválidos"]
if not (resumo_qualidade[contagens].eq(0).all().all()
        and resumo_qualidade["mesmas_colunas"].all()
        and resumo_qualidade["ordem_temporal"].all()):
    raise ValueError("Confira os problemas de qualidade antes do treino.")
resumo_qualidade.to_csv(exp.resultados / "resumo_qualidade.csv", index=False)


,arquivo,mesmas_colunas,valores_ausentes,horários_inválidos,horários_repetidos,ordem_temporal,rótulos_inválidos
0,train1.csv.gz,True,0,0,0,True,0
1,train2.csv.gz,True,0,0,0,True,0
2,train3.csv.gz,True,0,0,0,True,0
3,test1.csv.gz,True,0,0,0,True,0
4,test2.csv.gz,True,0,0,0,True,0
5,test3.csv.gz,True,0,0,0,True,0
6,test4.csv.gz,True,0,0,0,True,0
7,test5.csv.gz,True,0,0,0,True,0


In [7]:
for etapa, nomes in exp.config["particoes"].items():
    print(etapa, nomes)
exp.registrar("exploracao", arquivos=len(arquivos), colunas_originais=amostra.shape[1])


treino ['test1.csv.gz', 'test2.csv.gz', 'test3.csv.gz']
validacao ['test4.csv.gz']
teste ['test5.csv.gz']
